# <font color="#418FDE" size="6.5" uppercase>**Klassifikation vergleichen**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Trainieren lineare, probabilistische, nachbarschaftsbasierte, baumbasierte und SVM-Klassifikatoren. 
- Untersuchen Wahrscheinlichkeiten, Klassen- und Stichprobengewichte sowie Voting. 
- Vergleichen Klassifikatoren mit Entscheidungsgrenzen, Konfusionsmatrizen und Fehleranalysen. 


## **1. Lineare Klassifikatoren**

### **1.1. Logistische Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_01_01.jpg?v=1787645216" width="250">



>* Lineares Modell für Klassenwahrscheinlichkeiten
>* Schnell, interpretierbar und mit Entscheidungsgrenze

>* Parameter lernen Einflüsse auf Klassenwahrscheinlichkeiten
>* Lineare Muster erfordern gute Merkmalsvorbereitung

>* Interpretierbare Wahrscheinlichkeiten unterstützen Entscheidungen
>* Datenqualität, Kalibrierung und Regularisierung beachten



In [ ]:
#@title Python-Code - Logistische Regression

# Wir trainieren eine logistische Regression.
# Wahrscheinlichkeiten erklären die lineare Entscheidung.
# Die Grafik zeigt die gelernte Trennlinie.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine zweidimensionale Klassifikationsdaten.
features, target = make_classification(
    n_samples=160,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.4,
    random_state=42,
)

# Diese Prüfung macht die erwartete Datenform sichtbar.
if features.shape != (160, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Wir teilen Daten fair in Training und Test.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Die logistische Regression lernt eine lineare Grenze.
model = LogisticRegression(random_state=42, max_iter=300)
model.fit(X_train_scaled, y_train)

# Wir bewerten harte Klassen und Wahrscheinlichkeiten.
predicted_classes = model.predict(X_test_scaled)
predicted_probabilities = model.predict_proba(X_test_scaled)[:, 1]
accuracy = accuracy_score(y_test, predicted_classes)

# Drei kurze Ausgaben fassen das Ergebnis zusammen.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testgenauigkeit: {accuracy:.2f}")
print(f"Erste drei Wahrscheinlichkeiten für Klasse 1: {np.round(predicted_probabilities[:3], 2)}")

# Ein Gitter zeigt die vorhergesagte Wahrscheinlichkeit.
x_min = X_train_scaled[:, 0].min() - 0.8
x_max = X_train_scaled[:, 0].max() + 0.8
y_min = X_train_scaled[:, 1].min() - 0.8
y_max = X_train_scaled[:, 1].max() + 0.8

# Die Gitterpunkte werden wie normale Beispiele bewertet.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 120),
    np.linspace(y_min, y_max, 120),
)
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_probabilities = model.predict_proba(grid_points)[:, 1]

# Die Wahrscheinlichkeit 0,5 entspricht der Entscheidungsgrenze.
probability_surface = grid_probabilities.reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7, 5))
contour = ax.contourf(xx, yy, probability_surface, levels=20, cmap="RdBu", alpha=0.35)

# Trainingspunkte zeigen die beiden bekannten Klassen.
scatter = ax.scatter(
    X_train_scaled[:, 0],
    X_train_scaled[:, 1],
    c=y_train,
    cmap="bwr",
    edgecolor="black",
    s=45,
)

# Die schwarze Linie markiert die Modellentscheidung.
ax.contour(xx, yy, probability_surface, levels=[0.5], colors="black", linewidths=2)
ax.set_title("Logistische Regression: Wahrscheinlichkeit und Trennlinie")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")

# Die Legende hilft beim Lesen der Klassenfarben.
ax.legend(*scatter.legend_elements(), title="Klasse", loc="upper left")
plt.show()



### **1.2. SGD und Naive Bayes**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_01_02.jpg?v=1787645214" width="250">



>* SGD trainiert lineare Modelle schrittweise effizient
>* Skalierung und Lernrate beeinflussen Stabilität stark

>* Probabilistische Klassifikation mit Unabhängigkeitsannahme
>* Schnell, robust und stark bei Textdaten

>* SGD lernt Grenzen, Naive Bayes Wahrscheinlichkeiten
>* Beide liefern nützliche Baselines und Modellvergleiche



In [ ]:
#@title Python-Code - SGD und Naive Bayes

# Wir vergleichen SGD und Naive Bayes.
# Beide Modelle lernen einfache Klassifikationsregeln.
# Die Ausgabe zeigt Genauigkeit und Entscheidungsgrenzen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Ein kleiner Datensatz macht die Grenzen sichtbar.
features, target = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=42,
)

# Die Formprüfung verhindert stille Datenfehler.
if features.shape != (300, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Stratifikation erhält ähnliche Klassenanteile in beiden Teilen.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# SGD braucht skalierte Merkmale für stabile Lernschritte.
sgd_model = make_pipeline(
    StandardScaler(),
    SGDClassifier(loss="log_loss", max_iter=2000, tol=1e-3, random_state=42),
)

# GaussianNB schätzt klassenweise einfache Wahrscheinlichkeitsverteilungen.
nb_model = GaussianNB()
sgd_model.fit(X_train, y_train)
nb_model.fit(X_train, y_train)

# Genauigkeit zeigt einen ersten fairen Testvergleich.
sgd_accuracy = accuracy_score(y_test, sgd_model.predict(X_test))
nb_accuracy = accuracy_score(y_test, nb_model.predict(X_test))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"SGD-Testgenauigkeit: {sgd_accuracy:.2f}")
print(f"Naive-Bayes-Testgenauigkeit: {nb_accuracy:.2f}")

# Ein Gitter zeigt, wo beide Modelle Klasse eins erwarten.
x_min = features[:, 0].min() - 0.8
x_max = features[:, 0].max() + 0.8
y_min = features[:, 1].min() - 0.8

# Die zweite Achsengrenze ergänzt das Vorhersagegitter.
y_max = features[:, 1].max() + 0.8
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 160),
    np.linspace(y_min, y_max, 160),
)

# Vorhersagen auf dem Gitter ergeben Entscheidungsflächen.
grid_points = np.c_[xx.ravel(), yy.ravel()]
sgd_grid = sgd_model.predict(grid_points).reshape(xx.shape)
nb_grid = nb_model.predict(grid_points).reshape(xx.shape)

# Die Differenz markiert Bereiche mit unterschiedlicher Entscheidung.
difference = sgd_grid - nb_grid
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, difference, levels=[-1, -0.5, 0.5, 1], alpha=0.25)

# Die Punkte zeigen die echten Klassen im Datensatz.
scatter = ax.scatter(
    X_test[:, 0],
    X_test[:, 1],
    c=y_test,
    cmap="coolwarm",
    edgecolor="black",
    s=35,
)

# Beschriftungen machen die Darstellung für Anfänger lesbar.
ax.set_title("SGD und Naive Bayes: unterschiedliche Entscheidungen")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(*scatter.legend_elements(), title="Klasse")
plt.show()



### **1.3. LDA und QDA**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_01_03.jpg?v=1787645218" width="250">



>* LDA: gleiche Streuung, lineare Entscheidungsgrenzen
>* QDA: eigene Streuung, gekrümmte Entscheidungsgrenzen

>* LDA: stabil bei kleinen, ähnlichen Gruppen
>* QDA: flexibel bei großen, unterschiedlich streuenden Daten

>* LDA und QDA schätzen Klassenwahrscheinlichkeiten.
>* Annahmen prüfen, sonst werden Modelle instabil.



In [ ]:
#@title Python-Code - LDA und QDA

# Dieses Beispiel vergleicht LDA und QDA.
# Beide Modelle lernen aus denselben Merkmalen.
# Die Grafik zeigt unterschiedliche Entscheidungsgrenzen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Wir erzeugen kleine zweidimensionale Klassifikationsdaten.
features, target = make_classification(
    n_samples=240,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.0,
    random_state=42,
)

# Eine einfache Prüfung schützt vor unerwarteten Datenformen.
if features.shape != (240, 2) or target.shape[0] != 240:
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Der Testanteil bleibt für beide Modelle identisch.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# LDA nutzt eine gemeinsame Streuungsstruktur aller Klassen.
lda_model = LinearDiscriminantAnalysis()
lda_model.fit(X_train, y_train)

# QDA erlaubt jeder Klasse eine eigene Streuungsstruktur.
qda_model = QuadraticDiscriminantAnalysis()
qda_model.fit(X_train, y_train)

# Wir bewerten beide Modelle auf denselben Testdaten.
lda_accuracy = accuracy_score(y_test, lda_model.predict(X_test))
qda_accuracy = accuracy_score(y_test, qda_model.predict(X_test))

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"LDA Testgenauigkeit: {lda_accuracy:.2f}")
print(f"QDA Testgenauigkeit: {qda_accuracy:.2f}")

# Ein Gitter macht die Entscheidungsbereiche sichtbar.
x_min = features[:, 0].min() - 0.8
x_max = features[:, 0].max() + 0.8
y_min = features[:, 1].min() - 0.8
y_max = features[:, 1].max() + 0.8

# Die Gitterauflösung bleibt klein und schnell.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 180),
    np.linspace(y_min, y_max, 180),
)

# Vorhersagen auf dem Gitter zeigen Modellgrenzen.
grid_points = np.c_[xx.ravel(), yy.ravel()]
lda_grid = lda_model.predict(grid_points).reshape(xx.shape)
qda_grid = qda_model.predict(grid_points).reshape(xx.shape)

# Eine einzige Achse vergleicht beide Grenzen direkt.
fig, ax = plt.subplots(figsize=(7, 5))
ax.contour(xx, yy, lda_grid, levels=[0.5], colors="black", linewidths=2)
ax.contour(xx, yy, qda_grid, levels=[0.5], colors="red", linewidths=2)

# Die Punkte zeigen die ursprünglichen Klassen.
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=target,
    cmap="coolwarm",
    edgecolor="white",
    s=35,
)

# Beschriftungen erklären die sichtbaren Modellunterschiede.
ax.set_title("LDA linear, QDA gekrümmt")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")

# Die Legende benennt Punkte und Entscheidungsgrenzen.
ax.plot([], [], color="black", linewidth=2, label="LDA-Grenze")
ax.plot([], [], color="red", linewidth=2, label="QDA-Grenze")
ax.legend(loc="best")

plt.show()



## **2. Bäume und Ensembles**

### **2.1. Entscheidungsbaum**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_02_01.jpg?v=1787645220" width="250">



>* Baum teilt Daten durch einfache Fragen
>* Blätter liefern interpretierbare, unsichere Wahrscheinlichkeiten

>* Gewichte machen wichtige Fehler sichtbarer
>* Bäume wählen dadurch sinnvollere Aufteilungen

>* Einzelne Bäume sind verständlich, aber instabil
>* Voting und Ensembles stabilisieren Entscheidungen



In [ ]:
#@title Python-Code - Entscheidungsbaum

# Wir untersuchen Wahrscheinlichkeiten eines Entscheidungsbaums.
# Gewichte verändern die gelernte Klassengrenze sichtbar.
# Die Grafik zeigt ungewichtete und gewichtete Vorhersagen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

# Ein kleiner Datensatz macht die Wirkung der Gewichte sichtbar.
features, target = make_classification(
    n_samples=220,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    weights=[0.82, 0.18],
    class_sep=0.85,
    random_state=42,
)

# Diese Prüfung schützt vor unerwarteten Datenformen.
if features.shape != (220, 2) or target.shape[0] != 220:
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Zwei Bäume lernen dieselben Daten mit unterschiedlicher Fehlergewichtung.
plain_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
weighted_tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight={0: 1, 1: 4},
    random_state=42,
)

plain_tree.fit(features, target)
weighted_tree.fit(features, target)

# Wir betrachten Wahrscheinlichkeiten für dieselben neuen Beobachtungen.
probe_points = np.array([[-1.0, 0.0], [0.0, 0.0], [1.0, 0.0]])
plain_probabilities = plain_tree.predict_proba(probe_points)[:, 1]
weighted_probabilities = weighted_tree.predict_proba(probe_points)[:, 1]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Klasse 1 im Datensatz: {round(float(target.mean()), 2)}")
print(f"Ungewichtete Wahrscheinlichkeiten: {np.round(plain_probabilities, 2)}")
print(f"Gewichtete Wahrscheinlichkeiten: {np.round(weighted_probabilities, 2)}")

# Die Entscheidungsfläche zeigt, wo der gewichtete Baum Klasse 1 erwartet.
x_min = float(features[:, 0].min() - 0.5)
x_max = float(features[:, 0].max() + 0.5)
y_min = float(features[:, 1].min() - 0.5)
y_max = float(features[:, 1].max() + 0.5)

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 180),
    np.linspace(y_min, y_max, 180),
)

# Die Farbe ist die geschätzte Wahrscheinlichkeit für die seltenere Klasse.
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_probabilities = weighted_tree.predict_proba(grid_points)[:, 1]
grid_probabilities = grid_probabilities.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
contour = ax.contourf(xx, yy, grid_probabilities, levels=12, cmap="RdYlBu_r")
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=target,
    cmap="bwr",
    edgecolor="black",
    s=35,
)

ax.set_title("Gewichteter Entscheidungsbaum: Wahrscheinlichkeit für Klasse 1")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
fig.colorbar(contour, ax=ax, label="Geschätzte Wahrscheinlichkeit")
plt.show()



### **2.2. Random Forest**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_02_02.jpg?v=1787645222" width="250">



>* Viele zufällige Bäume betrachten unterschiedliche Datenaspekte
>* Gemeinsames Voting macht Vorhersagen stabiler

>* Baumstimmen schätzen Klassenwahrscheinlichkeiten.
>* Nützlich, aber nicht immer perfekt kalibriert.

>* Gewichte helfen bei seltenen wichtigen Klassen
>* Voting macht Entscheidungen robuster vergleichbar



In [ ]:
#@title Python-Code - Random Forest

# Wir untersuchen Random-Forest-Wahrscheinlichkeiten mit Gewichtung.
# Klassenungleichgewicht beeinflusst Stimmen und Vorhersagen.
# Die Grafik zeigt sichere und unsichere Bereiche.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

# Wir erzeugen kleine unausgewogene Klassifikationsdaten.
features, target = make_classification(
    n_samples=500,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    weights=[0.88, 0.12],
    class_sep=1.0,
    random_state=42,
)

# Diese Prüfung macht die erwartete Datenform sichtbar.
if features.shape != (500, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Wir teilen die Daten stratifiziert in Training und Test.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Klassengewichte betonen die seltene Klasse beim Lernen.
forest = RandomForestClassifier(
    n_estimators=120,
    max_depth=5,
    class_weight="balanced",
    random_state=42,
)

# Ein Random Forest lernt viele Bäume gemeinsam.
forest.fit(X_train, y_train)

# Wahrscheinlichkeiten entstehen aus den Stimmen der Bäume.
probabilities = forest.predict_proba(X_test)[:, 1]
predictions = forest.predict(X_test)
score = balanced_accuracy_score(y_test, predictions)

# Wir betrachten drei typische Testfälle.
order = np.argsort(np.abs(probabilities - 0.5))[:3]
shown_probs = [round(float(probabilities[index]), 2) for index in order]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Balanced Accuracy mit Klassengewichten: {score:.2f}")
print(f"Drei unsichere Wald-Wahrscheinlichkeiten: {shown_probs}")

# Für die Entscheidungsfläche berechnen wir ein kleines Raster.
x_min = features[:, 0].min() - 0.8
x_max = features[:, 0].max() + 0.8
y_min = features[:, 1].min() - 0.8
y_max = features[:, 1].max() + 0.8

# Jeder Rasterpunkt erhält eine Wahrscheinlichkeit für Klasse eins.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 120),
    np.linspace(y_min, y_max, 120),
)
grid = np.c_[xx.ravel(), yy.ravel()]
grid_probabilities = forest.predict_proba(grid)[:, 1].reshape(xx.shape)

# Die Grafik zeigt Voting-Wahrscheinlichkeiten und Trainingspunkte.
fig, ax = plt.subplots(figsize=(7, 5))
contour = ax.contourf(xx, yy, grid_probabilities, levels=12, cmap="RdYlBu_r")
scatter = ax.scatter(
    X_train[:, 0],
    X_train[:, 1],
    c=y_train,
    cmap="bwr",
    edgecolor="black",
    s=28,
)

ax.set_title("Random Forest: Wahrscheinlichkeit durch Baum-Voting")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
fig.colorbar(contour, ax=ax, label="Wahrscheinlichkeit für Klasse 1")
plt.show()



### **2.3. Boosting Ensembles**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_02_03.jpg?v=1787645223" width="250">



>* Einfache Modelle werden schrittweise stärker
>* Spätere Modelle korrigieren schwierige Fehler

>* Boosting-Wahrscheinlichkeiten zeigen Sicherheit, nicht nur Klassen
>* Kalibrierung ist für riskante Entscheidungen entscheidend

>* Gewichte betonen seltene, wichtige Fehlerfälle.
>* Boosting braucht Validierung gegen Überanpassung.



In [ ]:
#@title Python-Code - Boosting Ensembles

# Dieses Beispiel zeigt Boosting mit Stichprobengewichten.
# Schwierige Minderheitsfälle erhalten bewusst mehr Lerngewicht.
# Wahrscheinlichkeiten zeigen veränderte Modellaufmerksamkeit.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Wir erzeugen kleine unausgewogene Klassifikationsdaten.
features, target = make_classification(
    n_samples=500,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    weights=[0.88, 0.12],
    class_sep=0.9,
    random_state=42,
)

# Diese Prüfung macht die Datenannahme sichtbar.
if features.shape != (500, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Der Split bewahrt das Klassenverhältnis ungefähr.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# Minderheitsbeispiele bekommen im Training mehr Gewicht.
class_counts = np.bincount(y_train)
minority_weight = class_counts[0] / class_counts[1]
sample_weight = np.where(y_train == 1, minority_weight, 1.0)

# AdaBoost kombiniert viele sehr einfache Entscheidungsstümpfe.
base_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
boosting_model = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=40,
    learning_rate=0.5,
    random_state=42,
)

# Das Modell lernt mit den vorbereiteten Stichprobengewichten.
boosting_model.fit(X_train, y_train, sample_weight=sample_weight)
y_pred = boosting_model.predict(X_test)
y_proba = boosting_model.predict_proba(X_test)[:, 1]

# Drei kurze Kennzahlen verbinden Gewichtung und Wahrscheinlichkeiten.
balanced_acc = balanced_accuracy_score(y_test, y_pred)
mean_minority_proba = y_proba[y_test == 1].mean()
mean_majority_proba = y_proba[y_test == 0].mean()

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsanteil Klasse 1: {y_train.mean():.2f}")
print(f"Gewicht für Klasse 1: {minority_weight:.1f}")
print(f"Balanced Accuracy: {balanced_acc:.2f}")
print(f"Mittlere P(Klasse 1) für echte Klasse 1: {mean_minority_proba:.2f}")
print(f"Mittlere P(Klasse 1) für echte Klasse 0: {mean_majority_proba:.2f}")

# Die Farbe zeigt die vorhergesagte Wahrscheinlichkeit für Klasse 1.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    X_test[:, 0],
    X_test[:, 1],
    c=y_proba,
    cmap="viridis",
    s=45,
)

# Markierungen zeigen die tatsächlichen Klassen im Testset.
ax.scatter(
    X_test[y_test == 1, 0],
    X_test[y_test == 1, 1],
    facecolors="none",
    edgecolors="red",
    s=90,
    label="echte Klasse 1",
)

ax.set_title("AdaBoost: Wahrscheinlichkeiten nach Stichprobengewichtung")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(loc="best")
fig.colorbar(scatter, ax=ax, label="Vorhergesagte P(Klasse 1)")
plt.show()



## **3. SVM Bewertung Vergleich**

### **3.1. SVM Modelle vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_03_01.jpg?v=1787645208" width="250">



>* Lineare SVM für einfache Trennungen
>* Nichtlineare Kernel erfassen komplexe Grenzen

>* Fehlertoleranz beeinflusst Komplexität und Robustheit
>* Konfusionsmatrizen zeigen wichtige Fehlertypen

>* Fehler nahe der Grenze gezielt untersuchen
>* Stabilität und Interpretierbarkeit gemeinsam bewerten



In [ ]:
#@title Python-Code - SVM Modelle vergleichen

# Wir vergleichen zwei SVM-Modelle anschaulich.
# Entscheidungsgrenzen zeigen unterschiedliche Modellkomplexität.
# Testwerte und Fehlerpunkte machen Unterschiede sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Ein kleiner Datensatz erzeugt eine nichtlineare Klassifikationsaufgabe.
features, target = make_moons(n_samples=240, noise=0.25, random_state=42)

# Diese Prüfung macht die erwartete Datenform sichtbar.
if features.shape != (240, 2) or target.shape != (240,):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Die Aufteilung trennt Training und faire Bewertung.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.35, stratify=target, random_state=42
)

# Beide Modelle nutzen dieselbe Skalierung ohne Datenleck.
linear_svm = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0))
rbf_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale"))

# Jedes Modell lernt nur aus den Trainingsdaten.
linear_svm.fit(X_train, y_train)
rbf_svm.fit(X_train, y_train)

# Die Testdaten zeigen, wie gut neue Punkte klassifiziert werden.
linear_pred = linear_svm.predict(X_test)
rbf_pred = rbf_svm.predict(X_test)

# Fehlerpunkte helfen beim Vergleich der Modellentscheidungen.
linear_errors = int(np.sum(linear_pred != y_test))
rbf_errors = int(np.sum(rbf_pred != y_test))

# Drei kurze Kennzahlen fassen den Vergleich zusammen.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Lineare SVM: Genauigkeit {accuracy_score(y_test, linear_pred):.2f}, Fehler {linear_errors}")
print(f"RBF-SVM: Genauigkeit {accuracy_score(y_test, rbf_pred):.2f}, Fehler {rbf_errors}")

# Ein Gitter macht die Entscheidungsgrenzen sichtbar.
x_min = features[:, 0].min() - 0.5
x_max = features[:, 0].max() + 0.5
y_min = features[:, 1].min() - 0.5
y_max = features[:, 1].max() + 0.5

# Die Auflösung bleibt klein und schnell genug.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 180), np.linspace(y_min, y_max, 180)
)

# Die RBF-Grenze wird als Hintergrund dargestellt.
grid_points = np.c_[xx.ravel(), yy.ravel()]
rbf_grid = rbf_svm.predict(grid_points).reshape(xx.shape)

# Die lineare Grenze wird als Vergleichslinie ergänzt.
linear_score = linear_svm.decision_function(grid_points).reshape(xx.shape)

# Eine einzige Grafik verbindet Grenze, Daten und Fehler.
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, rbf_grid, alpha=0.22, levels=[-0.5, 0.5, 1.5])
ax.contour(xx, yy, linear_score, levels=[0], colors="black", linewidths=2)
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=28, alpha=0.75)

# Falsch klassifizierte Testpunkte der RBF-SVM werden markiert.
rbf_wrong = rbf_pred != y_test
ax.scatter(
    X_test[rbf_wrong, 0], X_test[rbf_wrong, 1], facecolors="none", edgecolors="red", s=90, label="RBF-Fehler"
)

# Beschriftungen erklären die sichtbaren Modellunterschiede.
ax.set_title("SVM-Vergleich: lineare Grenze gegen RBF-Hintergrund")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(loc="upper right")
plt.show()



### **3.2. Gewichte und Voting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_03_02.jpg?v=1787645212" width="250">



>* Gewichte helfen bei unausgeglichenen Klassen.
>* SVM-Grenzen verschieben sich zugunsten seltener Fälle.

>* Gewichtung verändert Sensitivität und Präzision
>* Kontext, Grenzen und Fehler gemeinsam prüfen

>* Voting kombiniert Modelle für stabilere Entscheidungen
>* Fehleranalyse prüft Nutzen und gemeinsame Schwächen



In [ ]:
#@title Python-Code - Gewichte und Voting

# Dieses Beispiel vergleicht Gewichtung und Voting.
# Eine seltene Klasse erhält mehr Bedeutung.
# Die Grafik zeigt verschobene Entscheidungsgrenzen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import balanced_accuracy_score

# Wir erzeugen einen kleinen unausgeglichenen Klassifikationsdatensatz.
features, target = make_classification(
    n_samples=500, n_features=2, n_redundant=0,
    n_informative=2, weights=[0.88, 0.12], class_sep=0.9,
    random_state=42
)

# Die Aufteilung bleibt durch Stratifikation ähnlich unausgeglichen.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.35, stratify=target, random_state=42
)

# Eine kurze Prüfung macht die Klassenverteilung sichtbar.
minority_count = int(np.sum(y_train == 1))
majority_count = int(np.sum(y_train == 0))

# Drei SVMs unterscheiden sich nur in Gewichtung und Kernel.
plain_svm = make_pipeline(StandardScaler(), SVC(kernel="linear", random_state=42))
weighted_svm = make_pipeline(
    StandardScaler(), SVC(kernel="linear", class_weight="balanced", random_state=42)
)

rbf_svm = make_pipeline(
    StandardScaler(), SVC(kernel="rbf", gamma="scale", class_weight="balanced", random_state=42)
)

# Das Voting kombiniert die drei Perspektiven zu einer Entscheidung.
voting_model = VotingClassifier(
    estimators=[("plain", plain_svm), ("weighted", weighted_svm), ("rbf", rbf_svm)],
    voting="hard", weights=[1, 2, 2]
)

# Wir trainieren nur das Ensemble; es trainiert seine Teilmodelle mit.
voting_model.fit(X_train, y_train)

# Die Teilmodelle im Ensemble können einzeln bewertet werden.
plain_score = balanced_accuracy_score(y_test, voting_model.named_estimators_["plain"].predict(X_test))
weighted_score = balanced_accuracy_score(y_test, voting_model.named_estimators_["weighted"].predict(X_test))
voting_score = balanced_accuracy_score(y_test, voting_model.predict(X_test))

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Trainingsdaten: Klasse 0 = {majority_count}, Klasse 1 = {minority_count}")
print(f"Balanced Accuracy ohne Gewichtung: {plain_score:.2f}")
print(f"Balanced Accuracy mit Gewichtung: {weighted_score:.2f}")
print(f"Balanced Accuracy mit gewichtetem Voting: {voting_score:.2f}")

# Für die Grafik berechnen wir die Voting-Entscheidung im Merkmalsraum.
x_min = features[:, 0].min() - 0.8
x_max = features[:, 0].max() + 0.8
y_min = features[:, 1].min() - 0.8
y_max = features[:, 1].max() + 0.8

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 180), np.linspace(y_min, y_max, 180))
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_prediction = voting_model.predict(grid_points).reshape(xx.shape)

# Eine einzige Grafik zeigt Datenpunkte und Voting-Grenze.
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, grid_prediction, levels=[-0.5, 0.5, 1.5], alpha=0.25)
scatter = ax.scatter(features[:, 0], features[:, 1], c=target, s=22, edgecolor="k")

ax.set_title("Gewichtetes Voting bei unausgeglichenen Klassen")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(*scatter.legend_elements(), title="Klasse", loc="upper right")
plt.show()



### **3.3. Klassifikatoren im Vergleich**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_03_03.jpg?v=1787645210" width="250">



>* Modelle nach Entscheidungen und Fehlern bewerten
>* Grenzen, Stabilität und Nutzbarkeit vergleichen

>* Entscheidungsgrenzen zeigen Modellunterschiede und Überanpassung
>* Konfusionsmatrizen bewerten Fehler im Anwendungskontext

>* Fehler zeigen Ursachen und Verbesserungsmöglichkeiten.
>* Verlässlichkeit zählt mehr als reine Gewinner.



In [ ]:
#@title Python-Code - Klassifikatoren im Vergleich

# Wir vergleichen Klassifikatoren auf denselben zweidimensionalen Daten.
# Entscheidungsgrenzen zeigen unterschiedliche Modellannahmen sichtbar.
# Die Ausgabe verbindet Genauigkeit und Fehlertypen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# Ein kleiner Datensatz macht Entscheidungsgrenzen gut sichtbar.
features, target = make_moons(
    n_samples=240,
    noise=0.25,
    random_state=42,
)

# Diese Prüfung verhindert unklare Fehler bei falschen Datenformen.
if features.shape != (240, 2):
    raise ValueError("Die Beispieldaten müssen 240 Zeilen und 2 Merkmale haben.")

# Die Aufteilung bleibt fair durch gleiche Klassenanteile.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# Eine lineare SVM ist einfach und gut interpretierbar.
model = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear", C=1.0, random_state=42),
)

# Das Modell lernt nur aus den Trainingsdaten.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Genauigkeit und Konfusionsmatrix zeigen verschiedene Perspektiven.
accuracy = accuracy_score(y_test, y_pred)
confusion = confusion_matrix(y_test, y_pred)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testgenauigkeit der linearen SVM: {accuracy:.2f}")
print(f"Konfusionsmatrix [[richtig 0, falsch 1], [falsch 0, richtig 1]]: {confusion.tolist()}")

# Ein Raster zeigt, wo das Modell Klasse 0 oder 1 erwartet.
x_min = features[:, 0].min() - 0.5
x_max = features[:, 0].max() + 0.5
y_min = features[:, 1].min() - 0.5
y_max = features[:, 1].max() + 0.5

# Die Rasterauflösung bleibt klein und schnell.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 160),
    np.linspace(y_min, y_max, 160),
)

# Für jeden Rasterpunkt wird eine Klasse vorhergesagt.
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_pred = model.predict(grid_points)
grid_pred = grid_pred.reshape(xx.shape)

# Eine einzige Grafik verbindet Datenpunkte und Entscheidungsgrenze.
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, grid_pred, alpha=0.25, cmap="coolwarm")
scatter = ax.scatter(
    X_test[:, 0],
    X_test[:, 1],
    c=y_test,
    cmap="coolwarm",
    edgecolor="black",
)

ax.set_title("Lineare SVM: Entscheidungsgrenze und Testpunkte")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(*scatter.legend_elements(), title="Wahre Klasse")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Klassifikation vergleichen**</font>


In this lecture, you learned to:
- Trainieren lineare, probabilistische, nachbarschaftsbasierte, baumbasierte und SVM-Klassifikatoren. 
- Untersuchen Wahrscheinlichkeiten, Klassen- und Stichprobengewichte sowie Voting. 
- Vergleichen Klassifikatoren mit Entscheidungsgrenzen, Konfusionsmatrizen und Fehleranalysen. 

In the next Module (Module 11), we will go over 'Bewertung und Tuning'